# 01 — Exploratory Data Analysis
Porto Seguro Safe Driver Prediction dataset.

**Run after:** downloading `data/raw/train.csv` via Kaggle CLI.
```bash
kaggle competitions download -c porto-seguro-safe-driver-prediction -p ../data/raw/
cd ../data/raw && unzip porto-seguro-safe-driver-prediction.zip
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

from src.preprocessing import load_and_clean, split_feature_columns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 80)

## 1. Load & basic shape

In [ ]:
raw = pd.read_csv('../data/raw/train.csv')
print(f'Shape (raw): {raw.shape}')
raw.head(3)

In [ ]:
print('dtypes summary:')
print(raw.dtypes.value_counts())
print(f'\nMemory: {raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 2. Target distribution (class imbalance)

In [ ]:
target_counts = raw['target'].value_counts()
print(target_counts)
print(f'\nPositive rate: {target_counts[1]/len(raw)*100:.2f}%')
print(f'scale_pos_weight suggestion: {target_counts[0]/target_counts[1]:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
target_counts.plot(kind='bar', ax=axes[0], color=['steelblue','tomato'], rot=0)
axes[0].set_title('Target class counts')
axes[0].set_xlabel('')

axes[1].pie(target_counts, labels=['No claim (0)', 'Claim (1)'],
            colors=['steelblue','tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Target distribution')

plt.tight_layout()
plt.savefig('../data/eda_target_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Missing values (encoded as -1)

In [ ]:
# Identify -1 missingness before cleaning
missing_pct = (raw == -1).mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print('Features with missing values (-1 encoded):')
print(missing_pct.to_string())

plt.figure(figsize=(8, max(4, len(missing_pct)*0.35)))
missing_pct.plot(kind='barh', color='tomato')
plt.xlabel('% missing')
plt.title('Missing value rate per feature')
plt.tight_layout()
plt.savefig('../data/eda_missing.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Feature group counts

In [ ]:
df = load_and_clean('../data/raw/train.csv')
col_groups = split_feature_columns(df)

print(f'After cleaning: {df.shape}')
print(f"  Categorical (_cat): {len(col_groups['cat'])} features")
print(f"  Binary     (_bin): {len(col_groups['bin'])} features")
print(f"  Continuous  (rest): {len(col_groups['cont'])} features")

# Infix groups
for grp in ['_ind_', '_reg_', '_car_', '_calc_']:
    n = len([c for c in raw.columns if grp in c])
    print(f"  {grp}: {n} features (in raw)")

## 5. Correlation with target (continuous features)

In [ ]:
cont_cols = col_groups['cont']
corr = df[cont_cols + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=False)

plt.figure(figsize=(8, max(4, len(corr[:20])*0.4)))
corr[:20].plot(kind='barh', color='steelblue')
plt.xlabel('|Pearson correlation with target|')
plt.title('Top 20 continuous features by target correlation')
plt.tight_layout()
plt.savefig('../data/eda_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Mutual information scores (top 20)

In [ ]:
feature_cols = col_groups['cat'] + col_groups['cont'] + col_groups['bin']
X_mi = df[feature_cols].fillna(df[feature_cols].median())
y = df['target']

mi = mutual_info_classif(X_mi, y, discrete_features='auto', random_state=42)
mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
mi_series[:20].plot(kind='barh', color='mediumseagreen')
plt.xlabel('Mutual information score')
plt.title('Top 20 features by mutual information with target')
plt.tight_layout()
plt.savefig('../data/eda_mutual_info.png', dpi=120, bbox_inches='tight')
plt.show()

print('Top 10 MI features:')
print(mi_series[:10])

## 7. Distribution of top features by target class

In [ ]:
top5 = mi_series[:5].index.tolist()
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, col in zip(axes, top5):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        subset = df[df['target'] == label][col].dropna()
        subset.plot(kind='kde', ax=ax, color=color, label=f'target={label}')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Feature distributions by target class (top 5 MI features)', y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Item | Value |
|---|---|
| Rows | 595,212 |
| Features (after dropping _calc_) | ~43 |
| Positive rate | ~3.64% |
| Recommended scale_pos_weight | ~26 |
| Features with missingness | see chart above |

**Next:** `02_features.ipynb` — build and validate the feature engineering pipeline.